In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import cv2
import numpy as np
import time
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. SETUP & EXACT PATHS
# ==========================================
# Enable CUDNN Benchmark for optimized convolution performance
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Initializing U-Net on device: {device}")

TRAIN_JSON = "/kaggle/input/datasets/hades1998/subsampling-json/subsampled_train_data.json"
TRAIN_DIR = "/kaggle/input/datasets/hrushi1998/vr-mini-proj-dataset/Trimmed_Dataset/trimmed_train_data"
VAL_DIR = "/kaggle/input/datasets/hrushi1998/vr-mini-proj-dataset/Trimmed_Dataset/trimmed_val_data"
VAL_JSON = os.path.join(VAL_DIR, "processed_val_data.json")

BEST_MODEL_PATH = "/kaggle/working/unet_resnet34_finetune.pth"

# 5 target classes + 1 Background = 6 Classes
NUM_CLASSES = 6 
IMG_SIZE = 512 # Standardizing to 512x512 for balance of speed and semantic resolution

# ==========================================
# 2. U-NET SEMANTIC DATASET CLASS 
# ==========================================
class SemanticJSONDataset(Dataset):
    def __init__(self, json_file, img_dir, img_size=512):
        self.img_dir = img_dir
        self.img_size = img_size
        with open(json_file, 'r') as f:
            self.full_data = json.load(f)["data"]
        
        self.top_5_categories = [1, 8, 7, 2, 9] 
        # Map specific category IDs to continuous integers: 1, 2, 3, 4, 5. Background is 0.
        self.label_map = {cat_id: idx + 1 for idx, cat_id in enumerate(self.top_5_categories)}

    def __len__(self): 
        return len(self.full_data)

    def __getitem__(self, idx):
        item = self.full_data[idx]
        img_path = os.path.join(self.img_dir, item["image_path"])
        if not os.path.exists(img_path): return None
            
        # Load Image
        image = cv2.imread(img_path)
        if image is None: return None
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h, w = image.shape[:2]
        
        # Create blank mask for Semantic Segmentation (0 = Background)
        mask = np.zeros((h, w), dtype=np.uint8)
        
        categories = item.get("item_categories", [])
        polygons = item.get("segmentation_polygons", [])
        
        valid_instances = 0
        for i in range(len(categories)):
            cat_id = categories[i]
            if cat_id in self.label_map:
                mapped_id = self.label_map[cat_id]
                valid_instances += 1
                
                # Rasterize polygons onto the semantic mask
                for poly in polygons[i]:
                    poly_pts = np.array(poly, dtype=np.int32).reshape((-1, 1, 2))
                    cv2.fillPoly(mask, [poly_pts], color=mapped_id)
                    
        if valid_instances == 0: return None
            
        # Resize: INTER_LINEAR for images, INTER_NEAREST for masks to preserve pure integers
        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        # To Tensor and Normalize (ImageNet stats for ResNet backbone)
        image_tensor = TF.to_tensor(image)
        image_tensor = TF.normalize(image_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask_tensor = torch.from_numpy(mask).long()

        return image_tensor, mask_tensor

def custom_collate(batch):
    batch = [b for b in batch if b is not None]
    if not batch: return torch.empty(0), torch.empty(0)
    images, masks = zip(*batch)
    return torch.stack(images), torch.stack(masks)

# ==========================================
# 3. U-NET ARCHITECTURE (ResNet-34)
# ==========================================
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels // 2 + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            diffY = skip.size()[2] - x.size()[2]
            diffX = skip.size()[3] - x.size()[3]
            x = F.pad(x, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
            x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class ResNet34UNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        resnet = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
        
        self.encoder0 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu) 
        self.maxpool = resnet.maxpool 
        self.encoder1 = resnet.layer1 
        self.encoder2 = resnet.layer2 
        self.encoder3 = resnet.layer3 
        self.encoder4 = resnet.layer4 

        self.decoder4 = DecoderBlock(in_channels=512, skip_channels=256, out_channels=256)
        self.decoder3 = DecoderBlock(in_channels=256, skip_channels=128, out_channels=128)
        self.decoder2 = DecoderBlock(in_channels=128, skip_channels=64,  out_channels=64)
        self.decoder1 = DecoderBlock(in_channels=64,  skip_channels=64,  out_channels=64)

        self.final_up = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        e0 = self.encoder0(x)       
        e1 = self.encoder1(self.maxpool(e0)) 
        e2 = self.encoder2(e1)      
        e3 = self.encoder3(e2)      
        bottleneck = self.encoder4(e3)

        d4 = self.decoder4(bottleneck, e3)
        d3 = self.decoder3(d4, e2)
        d2 = self.decoder2(d3, e1)
        d1 = self.decoder1(d2, e0)

        out = self.final_up(d1)
        out = self.final_conv(out)
        return out

# ==========================================
# 4. POST-PROCESSING (CCA & Voting)
# ==========================================
def post_process_instances(semantic_mask):
    """ Converts a 2D semantic mask into individual bounding boxes and instance labels. """
    binary_mask = np.uint8(semantic_mask > 0)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    instances = []
    for i in range(1, num_labels): # Skip 0 (Background)
        x, y, w, h, area = stats[i]
        component_pixels_mask = (labels == i)
        pixel_classes_in_component = semantic_mask[component_pixels_mask]
        
        # Majority voting for the class label
        majority_class = np.bincount(pixel_classes_in_component).argmax()
        
        instances.append({
            'instance_id': i,
            'bbox': [x, y, w, h],
            'class_label': majority_class,
            'area': area
        })
    return instances

# ==========================================
# 5. EXECUTION & TRAINING LOOP
# ==========================================
if __name__ == "__main__":
    # Dataloaders
    print("Loading datasets...")
    train_dataset = SemanticJSONDataset(TRAIN_JSON, TRAIN_DIR, img_size=IMG_SIZE)
    val_dataset = SemanticJSONDataset(VAL_JSON, VAL_DIR, img_size=IMG_SIZE)
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True, collate_fn=custom_collate)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True, collate_fn=custom_collate)

    # Initialize Model, Loss, and Optimizer
    model = ResNet34UNet(num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    
    epochs = 10
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler() # Mixed Precision Scaler
    
    best_val_loss = float('inf')

    print(f"\n🧠 Starting {epochs}-Epoch U-Net Fine-Tuning...")
    
    for epoch in range(epochs):
        epoch_start_time = time.time()
        model.train()
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [TRAIN]")
        train_loss = 0
        
        for images, masks in train_bar:
            if images.numel() == 0: continue
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True) # Slightly faster than zero_grad()
            
            # Autocast for Mixed Precision Training
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item()
            current_lr = optimizer.param_groups[0]['lr']
            train_bar.set_postfix({'Loss': f"{loss.item():.4f}", 'LR': f"{current_lr:.1e}"})
            
        avg_train_loss = train_loss / len(train_loader)
        scheduler.step()
        
        # Validation Phase
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, masks in val_loader:
                if images.numel() == 0: continue
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True)
                
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss = criterion(outputs, masks)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        epoch_time = int(time.time() - epoch_start_time)
        
        print(f" Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_time//60}m {epoch_time%60}s")
        
        if avg_val_loss < best_val_loss:
            print(f"🌟 New Best Validation Loss! Saving model to {BEST_MODEL_PATH}")
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), BEST_MODEL_PATH)

    print("\n✅ Training Complete! Testing Post-Processing on a validation sample...")
    
    # Quick Post-Processing Check
    model.load_state_dict(torch.load(BEST_MODEL_PATH, weights_only=True))
    model.eval()
    sample_img, _ = val_dataset[0]
    
    with torch.no_grad():
        out = model(sample_img.unsqueeze(0).to(device))
        pred_mask = torch.argmax(out, dim=1).squeeze(0).cpu().numpy()
        
    instances = post_process_instances(pred_mask)
    print(f"Test Image yielded {len(instances)} discrete instances via CCA and Majority Voting.")

🚀 Initializing U-Net on device: cuda
Loading datasets...
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 194MB/s]



🧠 Starting 10-Epoch U-Net Fine-Tuning...


Epoch 1/10 [TRAIN]: 100%|██████████| 4506/4506 [15:21<00:00,  4.89it/s, Loss=0.1575, LR=1.0e-04]


 Epoch 1 | Train Loss: 0.4583 | Val Loss: 0.3579 | Time: 20m 15s
🌟 New Best Validation Loss! Saving model to /kaggle/working/unet_resnet34_finetune.pth


...

 Epoch 8 | Train Loss: 0.0766 | Val Loss: 0.2567 | Time: 18m 30s


Epoch 9/10 [TRAIN]: 100%|██████████| 4506/4506 [14:39<00:00,  5.13it/s, Loss=0.0368, LR=1.0e-05]


 Epoch 9 | Train Loss: 0.0651 | Val Loss: 0.2702 | Time: 18m 21s


Epoch 10/10 [TRAIN]: 100%|██████████| 4506/4506 [14:38<00:00,  5.13it/s, Loss=0.1436, LR=3.4e-06]


 Epoch 10 | Train Loss: 0.0588 | Val Loss: 0.2712 | Time: 18m 22s

✅ Training Complete! Testing Post-Processing on a validation sample...
Test Image yielded 3 discrete instances via CCA and Majority Voting.
